In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

test= pd.read_csv('/content/drive/MyDrive/Irrigarion/test.csv')


class TorchNNClassifier(nn.Module):
     def __init__(self, model=None):
        super().__init__()
        self.model = model

     def forward(self, x):
        return self.model(x)

def Predicting_Irrigation_Need_PCA(input):

    
    
    extract_path = "/content/drive/MyDrive/deep_nn_model_PCA"
    feature_names = joblib.load(f"{extract_path}/feature_names.pkl")
    pipe = joblib.load(f"{extract_path}/pipeline_pca_nn.joblib")
    model = pipe.named_steps["nn"].model
    model.load_state_dict(torch.load(f"{extract_path}/model_weights.pth"))
    model.eval()

    new_df = pd.DataFrame([input])
    new_df = new_df.reindex(columns=feature_names, fill_value=0)
    X_np = new_df.values
    X_transformed = pipe.named_steps["pca"].transform(X_np)
    X_tensor = torch.tensor(X_transformed, dtype=torch.float32)

    with torch.no_grad():
        output = model(X_tensor)
        probs = F.softmax(output, dim=1)

        pred_class = torch.argmax(probs, dim=1).item()

    classes = ["Low", "Medium", "High"]

    print(classes[pred_class])

    return probs.numpy()[0]

In [4]:
input= test.iloc[10]
result= Predicting_Irrigation_Need_PCA(input)

Low
